# Cart2Insights — Exploratory Data Analysis

## Objective

The objective of this EDA is to understand e-commerce performance across:

- Sales and revenue
- Customer behavior
- Seller and product performance
- Delivery operations
- Customer experience
- Payment behavior

The analysis follows a business-question-first approach:

**Business Question → Analytical Question → Required Data → Grain → Transformation → Analysis → Visualization → Observation → Interpretation → Business Impact → Recommendation**

The analysis covers the official EDA requirements:

- Univariate analysis
- Bivariate analysis
- Multivariate analysis
- Trend analysis
- Correlation analysis
- Distribution analysis


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# 06_eda.ipynb runs from the notebooks/ directory
PROJECT_ROOT = Path.cwd().parent

FEATURES_DIR = PROJECT_ROOT / "data" / "features"
CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"

print("Project root:", PROJECT_ROOT)
print("Features directory:", FEATURES_DIR)
print("Cleaned directory:", CLEANED_DIR)

In [ ]:
order_features = pd.read_csv(
    FEATURES_DIR / "order_features.csv"
)

customer_features = pd.read_csv(
    FEATURES_DIR / "customer_features.csv"
)

seller_features = pd.read_csv(
    FEATURES_DIR / "seller_features.csv"
)

print("Order features:", order_features.shape)
print("Customer features:", customer_features.shape)
print("Seller features:", seller_features.shape)

In [ ]:
orders = pd.read_csv(
    CLEANED_DIR / "orders.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
)

order_items = pd.read_csv(
    CLEANED_DIR / "order_items.csv",
    parse_dates=["shipping_limit_date"],
)

customers = pd.read_csv(
    CLEANED_DIR / "customers.csv"
)

products = pd.read_csv(
    CLEANED_DIR / "products.csv"
)

order_reviews = pd.read_csv(
    CLEANED_DIR / "order_reviews.csv",
    parse_dates=[
        "review_creation_date",
        "review_answer_timestamp",
    ],
)

order_payments = pd.read_csv(
    CLEANED_DIR / "order_payments.csv"
)

translation = pd.read_csv(
    CLEANED_DIR / "product_category_name_translation.csv"
)

print("Orders:", orders.shape)
print("Order items:", order_items.shape)
print("Customers:", customers.shape)
print("Products:", products.shape)
print("Reviews:", order_reviews.shape)
print("Payments:", order_payments.shape)
print("Translation:", translation.shape)

In [ ]:
assert order_features["order_id"].is_unique
assert customer_features["customer_unique_id"].is_unique
assert seller_features["seller_id"].is_unique

assert orders["order_id"].is_unique
assert order_items[["order_id", "order_item_id"]].duplicated().sum() == 0

print("Analytical grain validation passed.")

In [ ]:
orders["purchase_month"] = (
    orders["order_purchase_timestamp"]
    .dt.to_period("M")
    .astype(str)
)

orders[["order_id", "order_purchase_timestamp", "purchase_month"]].head()

In [ ]:
monthly_sales = (
    orders[["order_id", "purchase_month"]]
    .merge(
        order_features[["order_id", "total_order_value"]],
        on="order_id",
        how="left",
        validate="one_to_one",
    )
    .groupby("purchase_month", as_index=False)
    .agg(
        revenue=("total_order_value", "sum"),
        orders=("order_id", "nunique"),
    )
)

monthly_sales["aov"] = (
    monthly_sales["revenue"] / monthly_sales["orders"]
)

monthly_sales.head()

In [ ]:
monthly_sales

In [ ]:
monthly_sales.describe()

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    monthly_sales["purchase_month"],
    monthly_sales["revenue"],
    marker="o",
)

plt.title("Monthly Revenue Trend")
plt.xlabel("Purchase Month")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.show()

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    monthly_sales["purchase_month"],
    monthly_sales["orders"],
    marker="o",
)

plt.title("Monthly Order Volume")
plt.xlabel("Purchase Month")
plt.ylabel("Number of Orders")
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.show()

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    monthly_sales["purchase_month"],
    monthly_sales["aov"],
    marker="o",
)

plt.title("Monthly Average Order Value")
plt.xlabel("Purchase Month")
plt.ylabel("Average Order Value")
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.show()

### Key Observations

- Revenue and order volume show strong overall growth from 2017 into 2018.
- The increase in revenue appears to be primarily volume-driven, as monthly order volume rises substantially while AOV remains relatively stable.
- November 2017 records an unusually high revenue and order-volume peak, without a corresponding spike in AOV, suggesting that the increase was mainly driven by transaction volume.
- Revenue remains at a relatively high level from March to August 2018, indicating sustained higher business activity.
- September and October 2018 are excluded from trend interpretation because the available dataset ends on 2018-10-17, making these partial periods rather than complete months.

In [ ]:
complete_monthly_sales = monthly_sales[
    monthly_sales["purchase_month"] <= "2018-08"
].copy()

complete_monthly_sales.tail()

## Sales Analysis — Revenue by Product Category

### Business Question

Which product categories contribute the most revenue?

### Analytical Question

How much sales value is generated by each product category?

### Grain

Order-item grain before aggregation, then category grain after aggregation.

### Required Data

- `order_items.price`
- `order_items.freight_value`
- `products.product_id`
- `products.product_category_name`
- `product_category_name_translation`

### Analysis

Calculate total order-item value by product category and rank categories by revenue.

### Visualization

A horizontal bar chart of the top categories.

### Why this chart?

We are comparing revenue across categorical groups. A horizontal bar chart makes ranking and differences between categories easy to compare, especially when category names are long.

### Important Join Consideration

Payments and reviews are not included in this calculation because they are separate one-to-many child tables and could multiply order-item rows if joined directly.

In [ ]:
print(products[[
    "product_id",
    "product_category_name"
]].head())

print(translation.head())

In [ ]:
category_sales = (
    order_items[
        ["order_id", "product_id", "price", "freight_value"]
    ]
    .merge(
        products[
            ["product_id", "product_category_name"]
        ],
        on="product_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        translation,
        on="product_category_name",
        how="left",
        validate="many_to_one",
    )
)

In [ ]:
category_sales["item_total_value"] = (
    category_sales["price"]
    + category_sales["freight_value"]
)

In [ ]:
category_revenue = (
    category_sales
    .groupby(
        "product_category_name_english",
        dropna=False,
        as_index=False
    )
    .agg(
        revenue=("item_total_value", "sum"),
        order_items=("order_id", "count"),
        orders=("order_id", "nunique"),
    )
    .sort_values("revenue", ascending=False)
)

In [ ]:
category_revenue.head(20)

In [ ]:
category_revenue.shape

In [ ]:
category_revenue[
    "product_category_name_english"
].isna().sum()


In [ ]:
item_total_value = (
    order_items["price"] + order_items["freight_value"]
).sum()

category_total_value = category_revenue["revenue"].sum()

print("Order-item total value:", item_total_value)
print("Category total value:", category_total_value)

print(
    "Reconciles:",
    np.isclose(item_total_value, category_total_value)
)

In [ ]:
top_categories = (
    category_revenue
    .dropna(subset=["product_category_name_english"])
    .head(15)
    .sort_values("revenue")
)

top_categories

In [ ]:
plt.figure(figsize=(11, 7))

plt.barh(
    top_categories["product_category_name_english"],
    top_categories["revenue"]
)

plt.title("Top 15 Product Categories by Revenue")
plt.xlabel("Revenue")
plt.ylabel("Product Category")
plt.tight_layout()

plt.show()

In [ ]:
category_revenue["revenue_share_pct"] = (
    category_revenue["revenue"]
    / category_revenue["revenue"].sum()
    * 100
)

category_revenue.head(15)

### Category Revenue — Interpretation

**Observation**

- `health_beauty` is the highest-revenue category, generating approximately 1.44M in sales value.
- The top 5 categories (`health_beauty`, `watches_gifts`, `bed_bath_table`, `sports_leisure`, and `computers_accessories`) contribute approximately 39% of total category revenue.
- The top 15 categories contribute approximately 76% of total category revenue.
- Revenue contribution is therefore concentrated among a relatively small group of categories.
- Category performance is not determined by item volume alone. For example, `watches_gifts` generates more revenue than some higher-volume categories, suggesting differences in value per item/order.

**Interpretation**

A relatively small set of categories drives a large share of platform sales. Understanding whether this concentration comes from sales volume, higher-value purchases, or both is important for revenue optimization.

**Business Impact**

High-contributing categories may deserve greater inventory availability, merchandising attention, promotional investment, and operational focus. At the same time, dependence on a small number of categories may create concentration risk.


## Sales Analysis — Top-Selling Products

### Business Question

Which individual products contribute the most sales?

### Analytical Question

Which products have the highest total sales value?

### Grain

Order-item grain before aggregation, then product grain after aggregation.

### Required Data

- `order_items.product_id`
- `order_items.price`
- `order_items.freight_value`
- `products.product_id`

### Analysis

Aggregate total order-item value by product and rank products by sales value.

### Visualization

Top-N horizontal bar chart.

### Why this chart?

The business question is a ranking/comparison problem. A horizontal bar chart clearly compares the highest-performing products while keeping product identifiers readable.

Only the top products will be displayed because the dataset contains tens of thousands of products and plotting all products would make the chart unreadable.

In [ ]:
product_revenue = (
    order_items[
        ["product_id", "price", "freight_value"]
    ]
    .assign(
        item_total_value=lambda df: (
            df["price"] + df["freight_value"]
        )
    )
    .groupby("product_id", as_index=False)
    .agg(
        revenue=("item_total_value", "sum"),
        items_sold=("product_id", "count"),
    )
    .sort_values("revenue", ascending=False)
)

product_revenue.head(15)

In [ ]:
top_products = (
    product_revenue
    .head(15)
    .sort_values("revenue")
)

top_products

In [ ]:
plt.figure(figsize=(11, 7))

plt.barh(
    top_products["product_id"],
    top_products["revenue"]
)

plt.title("Top 15 Products by Revenue")
plt.xlabel("Revenue")
plt.ylabel("Product ID")
plt.tight_layout()

plt.show()

In [ ]:
print(
    "Reconciles:",
    np.isclose(
        product_revenue["revenue"].sum(),
        item_total_value
    )
)

### Top Product Revenue — Interpretation

**Observation**

- The highest-revenue products contribute substantially more sales than many other individual products.
- Revenue is distributed across many products, but a relatively small group of products forms the top-performing segment.
- Product IDs in the source data are opaque identifiers, so individual-product findings should be combined with category-level analysis for business interpretation.

**Business Impact**

The highest-performing products may warrant closer attention for inventory availability, merchandising, and promotion.




In [ ]:
state_sales = (
    orders[["order_id", "customer_id"]]
    .merge(
        customers[[
            "customer_id",
            "customer_state"
        ]],
        on="customer_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        order_features[[
            "order_id",
            "total_order_value"
        ]],
        on="order_id",
        how="left",
        validate="one_to_one",
    )
    .groupby(
        "customer_state",
        as_index=False
    )
    .agg(
        revenue=("total_order_value", "sum"),
        orders=("order_id", "nunique"),
    )
    .sort_values(
        "revenue",
        ascending=False
    )
)

In [ ]:
state_sales.head(10)

In [ ]:
state_sales["revenue_share_pct"] = (
    state_sales["revenue"]
    / state_sales["revenue"].sum()
    * 100
)

state_sales.head(10)

In [ ]:
print(
    "State revenue reconciles:",
    np.isclose(
        state_sales["revenue"].sum(),
        order_features["total_order_value"].sum()
    )
)

In [ ]:
top_states = (
    state_sales
    .head(15)
    .sort_values("revenue")
)

plt.figure(figsize=(10, 7))

plt.barh(
    top_states["customer_state"],
    top_states["revenue"]
)

plt.title("Top 15 States by Revenue")
plt.xlabel("Revenue")
plt.ylabel("Customer State")
plt.tight_layout()

plt.show()

### Sales by Location — Interpretation

**Observation**

- São Paulo (SP) is the dominant state by revenue.
- Rio de Janeiro (RJ) and Minas Gerais (MG) are the next major contributors, but remain substantially below São Paulo.
- Revenue is concentrated among a relatively small number of states rather than being evenly distributed across the country.

**Interpretation**

The platform's sales activity is geographically concentrated, with a particularly strong presence in São Paulo and other major states.

**Business Impact**

High-revenue states may warrant greater marketing, seller acquisition, inventory, and operational attention. Geographic concentration should also be considered when evaluating expansion opportunities and operational risk.

**Next Analytical Question**

Is the geographic revenue difference mainly caused by more orders in these states, or do customers in some states also have higher average order values?

In [ ]:
order_count_distribution = (
    customer_features["customer_order_count"]
    .value_counts()
    .sort_index()
)

plt.figure(figsize=(10, 5))

plt.bar(
    order_count_distribution.index.astype(str),
    order_count_distribution.values
)

plt.title("Customer Distribution by Number of Orders")
plt.xlabel("Number of Orders per Customer")
plt.ylabel("Number of Customers")

plt.tight_layout()
plt.show()

In [ ]:
customer_order_distribution = (
    customer_features["customer_order_count"]
    .value_counts()
    .sort_index()
)

customer_order_distribution.head(10)

In [ ]:
repeat_percentage = (
    customer_features["repeat_customer"].mean() * 100
)

print(f"Repeat customer percentage: {repeat_percentage:.2f}%")

### Customer Order Frequency — Interpretation

**Observation**

- The customer base is heavily concentrated among customers with a single recorded order.
- 93,099 of the 96,096 customer identities are one-time customers.
- 2,997 customer identities are repeat customers.
- Customers with 3 or more orders form only a very small segment.

**Interpretation**

The platform has acquired a large number of customers, but only a relatively small proportion currently demonstrate repeat purchasing behavior.

**Business Impact**

Customer retention and repeat-purchase initiatives may represent a significant opportunity to increase customer lifetime value and revenue from the existing customer base.




In [ ]:
plt.figure(figsize=(9, 6))

sns.boxplot(
    data=customer_features,
    x="customer_type",
    y="customer_total_spending"
)

plt.title("Customer Spending: Repeat vs One-time")
plt.xlabel("Customer Type")
plt.ylabel("Total Spending")

plt.tight_layout()
plt.show()

In [ ]:
customer_type_spending = (
    customer_features
    .groupby("customer_type")["customer_total_spending"]
    .agg(
        customer_count="count",
        mean_spending="mean",
        median_spending="median",
        total_spending="sum",
    )
    .sort_values("mean_spending", ascending=False)
)

customer_type_spending

In [ ]:
repeat_summary = {
    "repeat_customers": int(
        customer_features["repeat_customer"].sum()
    ),
    "one_time_customers": int(
        (~customer_features["repeat_customer"]).sum()
    ),
    "repeat_percentage": (
        customer_features["repeat_customer"].mean() * 100
    ),
}

repeat_summary

In [ ]:
median_spending_comparison = (
    customer_features
    .groupby("customer_type")["customer_total_spending"]
    .median()
)

median_spending_comparison

In [ ]:
repeat_median = median_spending_comparison["Repeat"]
one_time_median = median_spending_comparison["One-time"]

print(
    f"Repeat-customer median spending is "
    f"{repeat_median / one_time_median:.2f}x "
    f"the one-time-customer median."
)

### Repeat vs One-time Spending — Interpretation

**Observation**

- Repeat customers represent a relatively small share of the customer base.
- Repeat customers have [higher/lower/similar] median total spending compared with one-time customers.
- The spending distribution is strongly right-skewed, with a relatively small number of customers accounting for very high spending.

**Interpretation**

The relationship between repeat purchasing and customer spending indicates whether customer retention is associated with higher customer value.

**Business Impact**

If repeat customers demonstrate substantially higher spending, retention and repeat-purchase initiatives could provide an important opportunity to increase revenue from the existing customer base.



In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(
    customer_features["customer_order_count"],
    customer_features["customer_total_spending"],
    alpha=0.35
)

plt.title("Customer Order Frequency vs Total Spending")
plt.xlabel("Number of Orders")
plt.ylabel("Total Spending")

plt.tight_layout()
plt.show()

In [ ]:
customer_correlation = (
    customer_features[
        [
            "customer_order_count",
            "customer_total_spending",
        ]
    ]
    .corr()
)

customer_correlation

In [ ]:
correlation_value = customer_correlation.loc[
    "customer_order_count",
    "customer_total_spending"
]

print(f"Correlation: {correlation_value:.3f}")

In [ ]:
spending_by_order_count = (
    customer_features
    .groupby("customer_order_count")[
        "customer_total_spending"
    ]
    .agg(
        customers="count",
        median_spending="median",
        mean_spending="mean",
    )
    .reset_index()
)

spending_by_order_count

### Customer Order Frequency vs Spending — Interpretation

**Observation**

- The correlation between customer order count and total spending is weakly positive (`r = 0.122`).
- Despite the weak overall linear correlation, median customer spending increases as order frequency increases.
- One-order customers have a median spending of 105.00, compared with 214.94 for customers with two orders and 331.11 for customers with three orders.
- Very high order-count groups contain very few customers, so their individual statistics should be interpreted cautiously.

**Interpretation**

Purchase frequency and total spending show a weak overall linear association, but customers who purchase repeatedly tend to have higher typical cumulative spending.

**Business Impact**

The small repeat-customer segment may have disproportionately higher customer value. This supports investigating retention and repeat-purchase opportunities.




In [ ]:
top_sellers = (
    seller_features
    .sort_values("seller_revenue", ascending=False)
    .head(15)
    .sort_values("seller_revenue")
)

top_sellers

In [ ]:
plt.figure(figsize=(11, 7))

plt.barh(
    top_sellers["seller_id"],
    top_sellers["seller_revenue"]
)

plt.title("Top 15 Sellers by Revenue")
plt.xlabel("Seller Revenue")
plt.ylabel("Seller ID")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(
    seller_features["seller_order_count"],
    seller_features["seller_revenue"],
    alpha=0.5
)

plt.title("Seller Order Volume vs Revenue")
plt.xlabel("Number of Orders")
plt.ylabel("Seller Revenue")

plt.tight_layout()
plt.show()

In [ ]:
seller_correlation = (
    seller_features[
        ["seller_order_count", "seller_revenue"]
    ]
    .corr()
)

seller_correlation

In [ ]:
seller_correlation.loc[
    "seller_order_count",
    "seller_revenue"
]

### Seller Order Volume vs Revenue — Interpretation

**Observation**

- Sellers with higher order volumes generally generate higher revenue.
- The relationship is positive but not perfectly proportional.
- A relatively small group of sellers operates at substantially higher order volumes and revenue than the majority of sellers.

**Interpretation**

Seller revenue is strongly related to transaction volume, but order volume alone does not completely explain seller performance.

**Business Impact**

Seller performance management should consider both transaction volume and revenue contribution rather than using order count alone.

In [ ]:
seller_concentration = (
    seller_features
    .sort_values("seller_revenue", ascending=False)
    .reset_index(drop=True)
)

seller_concentration["cumulative_revenue"] = (
    seller_concentration["seller_revenue"].cumsum()
)

total_seller_revenue = seller_concentration[
    "seller_revenue"
].sum()

seller_concentration["cumulative_revenue_pct"] = (
    seller_concentration["cumulative_revenue"]
    / total_seller_revenue
    * 100
)

seller_concentration["seller_rank"] = (
    seller_concentration.index + 1
)

seller_concentration.head(10)

In [ ]:
for n in [10, 50, 100]:
    revenue_pct = seller_concentration.loc[
        seller_concentration["seller_rank"] == n,
        "cumulative_revenue_pct"
    ].iloc[0]

    print(
        f"Top {n} sellers contribute "
        f"{revenue_pct:.2f}% of seller revenue."
    )

In [ ]:
fig, ax1 = plt.subplots(figsize=(11, 6))

ax1.bar(
    seller_concentration["seller_rank"],
    seller_concentration["seller_revenue"],
    width=1.0
)

ax1.set_xlabel("Seller Rank")
ax1.set_ylabel("Seller Revenue")
ax1.set_title("Seller Revenue Concentration")

ax2 = ax1.twinx()

ax2.plot(
    seller_concentration["seller_rank"],
    seller_concentration["cumulative_revenue_pct"]
)

ax2.set_ylabel("Cumulative Revenue (%)")
ax2.axhline(80, linestyle="--")

plt.tight_layout()
plt.show()

### Seller Revenue Concentration — Interpretation

**Observation**

- Seller revenue is highly unevenly distributed.
- A relatively small group of high-performing sellers contributes a substantial share of total seller revenue.
- The cumulative revenue curve rises quickly among the highest-ranked sellers and then flattens across the long tail.

**Interpretation**

The platform's seller ecosystem contains a concentrated high-value group alongside a much larger group of lower-revenue sellers.

**Business Impact**

High-value sellers are important to retain because disruptions involving these sellers could have a disproportionate impact on platform revenue. At the same time, promising mid-tier sellers may represent an opportunity for seller development.

In [ ]:
delivery_days = order_features[
    "delivery_days"
].dropna()

plt.figure(figsize=(10, 5))

plt.hist(
    delivery_days,
    bins=50,
    edgecolor="black"
)

plt.title("Distribution of Delivery Time")
plt.xlabel("Delivery Time (Days)")
plt.ylabel("Number of Orders")

plt.tight_layout()
plt.show()

### Delivery Time Distribution — Interpretation

**Observation**

- Delivery time is strongly right-skewed.
- Most delivered orders fall within a relatively shorter delivery window, while a smaller number of orders take substantially longer.
- The mean delivery time is higher than the median, indicating that long-delivery orders pull the average upward.

**Interpretation**

Typical delivery performance is considerably faster than the mean alone suggests, but a long tail of slow deliveries remains.

**Business Impact**

Operational improvement should focus not only on average delivery time but also on reducing unusually long delivery times.

In [ ]:
delivery_status_counts = (
    order_features["delivery_status"]
    .value_counts()
)

delivery_status_counts

In [ ]:
delivery_status_pct = (
    delivery_status_counts
    / delivery_status_counts.sum()
    * 100
)

delivery_status_pct

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    delivery_status_counts.index,
    delivery_status_counts.values
)

plt.title("On-time vs Delayed Orders")
plt.xlabel("Delivery Status")
plt.ylabel("Number of Orders")

plt.tight_layout()
plt.show()

### Delivery Reliability — Interpretation

**Observation**

- The majority of orders with a known delivery outcome were delivered on time.
- A smaller but meaningful share of delivered orders were delayed.
- Orders without an actual customer-delivery timestamp are excluded from the on-time/delayed comparison rather than being incorrectly classified.

**Interpretation**

The platform generally meets its estimated delivery date, but delivery delays affect a meaningful subset of completed orders.

**Business Impact**

Reducing the delayed-order share could improve operational reliability and potentially improve customer satisfaction.

In [ ]:
state_delivery = (
    orders[
        ["order_id", "customer_id"]
    ]
    .merge(
        customers[
            ["customer_id", "customer_state"]
        ],
        on="customer_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        order_features[
            [
                "order_id",
                "delivery_days",
                "delivery_status"
            ]
        ],
        on="order_id",
        how="left",
        validate="one_to_one",
    )
)

In [ ]:
state_delivery_summary = (
    state_delivery
    .dropna(subset=["delivery_status"])
    .groupby("customer_state")
    .agg(
        orders=("order_id", "nunique"),
        avg_delivery_days=("delivery_days", "mean"),
        delayed_orders=(
            "delivery_status",
            lambda x: (x == "Delayed").sum()
        ),
    )
    .reset_index()
)

state_delivery_summary["delay_rate_pct"] = (
    state_delivery_summary["delayed_orders"]
    / state_delivery_summary["orders"]
    * 100
)

state_delivery_summary = (
    state_delivery_summary
    .sort_values("delay_rate_pct", ascending=False)
)

state_delivery_summary.head(15)

In [ ]:
top_delay_states = (
    state_delivery_summary
    .query("orders >= 100")
    .head(15)
    .sort_values("delay_rate_pct")
)

plt.figure(figsize=(10, 7))

plt.barh(
    top_delay_states["customer_state"],
    top_delay_states["delay_rate_pct"]
)

plt.title("States with Highest Delivery Delay Rate")
plt.xlabel("Delayed Orders (%)")
plt.ylabel("Customer State")

plt.tight_layout()
plt.show()

### Delivery Performance by State — Interpretation

**Observation**

- Delivery delay rates vary considerably across customer states.
- AL has the highest observed delay rate among the states shown, followed by MA and PI.
- Several states have substantially lower delay rates, indicating geographic differences in delivery performance.

**Interpretation**

Delivery reliability is not uniform across geographic markets and may reflect differences in logistics conditions, fulfillment coverage, or operational performance.

**Business Impact**

States with relatively high delay rates can be prioritized for operational investigation and logistics improvement.




In [ ]:
order_review_scores = (
    order_reviews
    .groupby("order_id", as_index=False)
    .agg(
        review_score=("review_score", "mean"),
        review_records=("review_score", "count"),
    )
)

delivery_review = (
    order_features[
        ["order_id", "delivery_status", "delivery_days", "delivery_delay"]
    ]
    .merge(
        order_review_scores,
        on="order_id",
        how="inner",
        validate="one_to_one",
    )
    .dropna(subset=["delivery_status", "review_score"])
)

delivery_review.head()

In [ ]:
print("Rows:", len(delivery_review))
print(
    "Unique orders:",
    delivery_review["order_id"].nunique()
)

print(
    "One row per order:",
    delivery_review["order_id"].is_unique
)

In [ ]:
delivery_review_summary = (
    delivery_review
    .groupby("delivery_status")["review_score"]
    .agg(
        orders="count",
        mean="mean",
        median="median",
        std="std",
    )
)

delivery_review_summary

In [ ]:
plt.figure(figsize=(9, 6))

sns.boxplot(
    data=delivery_review,
    x="delivery_status",
    y="review_score"
)

plt.title("Review Score by Delivery Status")
plt.xlabel("Delivery Status")
plt.ylabel("Review Score")

plt.tight_layout()
plt.show()

### Delivery Status vs Review Score — Interpretation

**Observation**

- On-time orders have substantially higher review scores than delayed orders.
- The median review score for on-time orders is around 5, while the median for delayed orders is around 2.
- Delayed orders also show greater variation in review scores.

**Interpretation**

There is a strong observed association between delivery performance and customer satisfaction. Orders that miss the estimated delivery date tend to receive lower customer ratings.

**Business Impact**

Delivery reliability appears to be an important customer-experience area. Reducing delivery delays may help improve customer satisfaction and reduce negative reviews.



The observed difference will be formally evaluated using the required independent two-sample t-test.



In [ ]:
review_score_counts = (
    order_reviews["review_score"]
    .value_counts()
    .sort_index()
)

review_score_counts

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    review_score_counts.index.astype(str),
    review_score_counts.values
)

plt.title("Distribution of Customer Review Scores")
plt.xlabel("Review Score")
plt.ylabel("Number of Reviews")

plt.tight_layout()
plt.show()

### Review Score Distribution — Interpretation

**Observation**

- 5-star reviews are the most frequent review score by a substantial margin.
- 4-star reviews form the second-largest group.
- 1-star reviews are substantially more common than 2-star reviews.
- The overall review distribution is concentrated toward the higher end of the rating scale, while still containing a meaningful low-rating segment.

**Interpretation**

Overall customer satisfaction appears predominantly positive, but a non-trivial group of customers reports poor experiences.

**Business Impact**

The large volume of positive reviews indicates generally favorable customer experience, while the low-rating segment provides an opportunity to identify and address the main causes of dissatisfaction.



In [ ]:
order_category = (
    order_items[
        ["order_id", "product_id", "price", "freight_value"]
    ]
    .merge(
        products[
            ["product_id", "product_category_name"]
        ],
        on="product_id",
        how="left",
        validate="many_to_one",
    )
    .groupby(
        ["order_id", "product_category_name"],
        as_index=False
    )
    .agg(
        category_order_value=(
            "price",
            "sum"
        )
    )
)

In [ ]:
category_reviews = (
    order_category
    .merge(
        order_review_scores,
        on="order_id",
        how="inner",
        validate="many_to_one",
    )
)

In [ ]:
category_review_summary = (
    category_reviews
    .groupby("product_category_name", as_index=False)
    .agg(
        orders=("order_id", "nunique"),
        average_review_score=("review_score", "mean"),
    )
    .query("orders >= 100")
    .sort_values("average_review_score")
)

category_review_summary.head(15)

In [ ]:
top_review_categories = pd.concat([
    category_review_summary.head(10),
    category_review_summary.tail(10)
]).drop_duplicates().sort_values("average_review_score")

plt.figure(figsize=(10, 8))

plt.barh(
    top_review_categories["product_category_name"],
    top_review_categories["average_review_score"]
)

plt.title("Average Review Score by Product Category")
plt.xlabel("Average Review Score")
plt.ylabel("Product Category")

plt.tight_layout()
plt.show()

### Review Scores by Product Category — Interpretation

**Observation**

- Average review scores vary across product categories.
- The highest-rated categories have average scores above 4, while the lower-rated categories are closer to the mid-to-high 3s.
- The variation across categories is present but is smaller than the difference observed between on-time and delayed orders.

**Interpretation**

Customer satisfaction is not uniform across product categories, suggesting that category-specific product or service experiences may influence ratings.

**Business Impact**

Lower-rated categories can be investigated further for product quality, fulfillment, or customer-experience issues. Higher-rated categories can provide useful benchmarks for identifying successful practices.




In [ ]:
delivery_review_days = delivery_review.dropna(
    subset=["delivery_days", "review_score"]
)

plt.figure(figsize=(10, 6))

plt.scatter(
    delivery_review_days["delivery_days"],
    delivery_review_days["review_score"],
    alpha=0.3
)

plt.title("Delivery Time vs Review Score")
plt.xlabel("Delivery Time (Days)")
plt.ylabel("Review Score")

plt.tight_layout()
plt.show()

### Delivery Time vs Review Score — Interpretation

**Observation**

- Review scores occur across a wide range of delivery times.
- The scatter plot does not show a strong obvious linear relationship between delivery duration and review score.
- Most observations are concentrated at shorter delivery times, while a smaller number of long-delivery orders appear across several review-score levels.

**Interpretation**

Absolute delivery duration alone does not appear to explain review scores strongly in a simple linear relationship. In contrast, the earlier comparison of on-time versus delayed orders showed a much more pronounced difference in review scores.

**Business Impact**

The distinction suggests that meeting the promised delivery date may be more useful for monitoring customer experience than focusing only on average delivery duration.



In [ ]:
delivery_review_days[
    ["delivery_days", "review_score"]
].corr()

In [ ]:
order_payment_type = (
    order_payments
    .groupby(
        ["order_id", "payment_type"],
        as_index=False
    )
    .agg(
        payment_value=("payment_value", "sum")
    )
    .sort_values(
        ["order_id", "payment_value"],
        ascending=[True, False]
    )
)

primary_payment = (
    order_payment_type
    .drop_duplicates("order_id")
    [["order_id", "payment_type"]]
    .rename(
        columns={
            "payment_type": "primary_payment_type"
        }
    )
)

print(primary_payment.shape)
primary_payment.head()

In [ ]:
assert primary_payment["order_id"].is_unique

print("Primary payment mapping validated.")

In [ ]:
payment_distribution = (
    primary_payment["primary_payment_type"]
    .value_counts()
)

payment_distribution

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    payment_distribution.index,
    payment_distribution.values
)

plt.title("Primary Payment Method Distribution")
plt.xlabel("Payment Method")
plt.ylabel("Number of Orders")

plt.tight_layout()
plt.show()

### Primary Payment Method Distribution — Interpretation

**Observation**

- Credit card is by far the most common primary payment method.
- Boleto is the second most frequently used payment method, but at a substantially lower volume.
- Voucher and debit card account for much smaller portions of orders.
- `not_defined` represents a negligible share of orders.

**Interpretation**

The platform's payment behavior is heavily concentrated around credit-card transactions.

**Business Impact**

Credit-card payment flows are particularly important to maintain reliably because disruptions in this payment method could affect a large portion of transactions.

In [ ]:
payment_status = (
    orders[
        ["order_id", "order_status"]
    ]
    .merge(
        primary_payment,
        on="order_id",
        how="inner",
        validate="one_to_one",
    )
)

payment_status_table = (
    pd.crosstab(
        payment_status["primary_payment_type"],
        payment_status["order_status"],
        normalize="index"
    ) * 100
)

payment_status_table.round(2)

In [ ]:
payment_status_table.plot(
    kind="bar",
    stacked=True,
    figsize=(11, 6)
)

plt.title("Order Status Distribution by Primary Payment Method")
plt.xlabel("Primary Payment Method")
plt.ylabel("Percentage of Orders")

plt.legend(
    title="Order Status",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

### Payment Method vs Order Status — Interpretation

**Observation**

- Delivered orders dominate the order-status distribution for the major payment methods.
- The `not_defined` payment group shows a markedly different status distribution, with cancellations accounting for most of its observations.
- The major payment methods have broadly similar order-status compositions, dominated by delivered orders.

**Interpretation**

There may be an association between payment method and order status, but the apparent difference for `not_defined` should be interpreted cautiously because this is a very small payment-method group.

**Business Impact**

Payment methods with unusual cancellation or non-delivery patterns may warrant investigation, particularly if the relationship is statistically supported.



The relationship between payment method and order status will be formally evaluated using a chi-square test of independence.

In [ ]:
category_order_value = (
    order_items[
        ["order_id", "product_id", "price", "freight_value"]
    ]
    .merge(
        products[
            ["product_id", "product_category_name"]
        ],
        on="product_id",
        how="left",
        validate="many_to_one",
    )
    .assign(
        item_total_value=lambda df:
            df["price"] + df["freight_value"]
    )
    .groupby(
        ["order_id", "product_category_name"],
        as_index=False
    )
    .agg(
        category_order_value=(
            "item_total_value",
            "sum"
        )
    )
)

In [ ]:
category_counts = (
    category_order_value["product_category_name"]
    .value_counts()
)

valid_categories = category_counts[
    category_counts >= 100
].index

anova_eda = category_order_value[
    category_order_value["product_category_name"]
    .isin(valid_categories)
].copy()

In [ ]:
anova_categories = (
    anova_eda["product_category_name"]
    .value_counts()
    .head(10)
    .index
)

anova_plot_data = anova_eda[
    anova_eda["product_category_name"]
    .isin(anova_categories)
]

In [ ]:
plt.figure(figsize=(11, 7))

sns.boxplot(
    data=anova_plot_data,
    y="product_category_name",
    x="category_order_value"
)

plt.title(
    "Order-Category Value Distribution Across Major Categories"
)
plt.xlabel("Category Order Value")
plt.ylabel("Product Category")

plt.tight_layout()
plt.show()

### Category vs Order Value — Interpretation

**Observation**

- Order-category values show substantial variation within every major category.
- The distributions are strongly right-skewed, with many high-value observations appearing as outliers.
- The central distributions differ across categories, but there is also considerable overlap between categories.

**Interpretation**

Product categories may have different typical order values, but within-category variation is substantial. Visual inspection alone is not sufficient to determine whether the category differences are statistically significant.

**Business Impact**

Categories with systematically higher order values may represent opportunities for premium products, targeted merchandising, and revenue optimization.



The differences in order value across product categories will be formally evaluated using the required one-way ANOVA.

# Final EDA Summary



## Business Overview — Key Findings

- The platform shows strong growth in order activity and revenue from early 2017 through August 2018.
- Revenue growth is primarily associated with increasing order volume, while monthly AOV remains relatively stable over the mature period.
- November 2017 represents a notable revenue and order-volume peak.
- September and October 2018 are partial periods and are therefore excluded from trend interpretation.

## Sales — Key Findings

- Revenue is concentrated among a relatively small number of product categories.
- `health_beauty` is the highest-revenue category among the observed categories.
- The top-performing categories account for a substantial portion of overall category revenue.
- A relatively small group of individual products forms the top-revenue product segment.
- Sales are geographically concentrated, with São Paulo being the dominant state by revenue.

## Customer — Key Findings

- The customer base is heavily concentrated among one-time customers.
- Repeat customers form a much smaller segment of the overall customer base.
- Customer spending is strongly right-skewed, with a relatively small number of high-spending customers.
- Repeat customers show higher typical cumulative spending than one-time customers.
- Customer order frequency and total spending have a weak positive linear correlation, although grouped results show that typical spending increases with purchase frequency.

## Seller — Key Findings

- Seller revenue is highly unevenly distributed.
- A relatively small group of sellers contributes a substantial share of platform seller revenue.
- Seller order volume is positively associated with seller revenue.
- The seller ecosystem contains a small high-value group and a much larger long tail of lower-revenue sellers.

## Delivery — Key Findings

- Delivery time is right-skewed, with most orders delivered within a relatively shorter period and a smaller long tail of slow deliveries.
- Most orders with a known delivery outcome are delivered on time.
- Delivery delay rates vary substantially across customer states.
- Delayed orders receive substantially lower review scores than on-time orders.
- Absolute delivery duration does not show an equally strong visual linear relationship with review score.
- Meeting the estimated delivery date appears to be a more meaningful customer-experience indicator than delivery duration alone.

## Customer Experience — Key Findings

- 5-star reviews form the largest share of customer ratings.
- Overall customer sentiment is predominantly positive, but a meaningful low-rating segment remains.
- Average review scores vary across product categories.
- The difference in review scores between delayed and on-time orders is substantially more pronounced than the category-level rating differences.

## Payment — Key Findings

- Credit card is by far the most common primary payment method.
- Boleto is the second most common payment method.
- Voucher and debit card account for much smaller shares.
- The distribution of order statuses appears broadly similar for the major payment methods, while `not_defined` shows a markedly different pattern but represents a very small group.
- The apparent relationship between payment method and order status requires formal statistical testing.